# 04 · dynamics_analysis — featurize → reduce → free-energy surface

`pip install "kinapse[structures,dynamics]"`

Turn a conformational ensemble (MD or generated) into features, reduce to 2-D, and build
a **free-energy surface (PMF)**. Comparing two ensembles' PMFs with **Jensen–Shannon
divergence** is the project's headline model-quality metric.

> `example_ensemble.xtc` is a small synthetic ensemble so the notebook runs; use your own MD.

In [ ]:
import os
os.environ.setdefault("ANARCI_CPU", "1")   # number on CPU (safe without a GPU)

from kinapse.structures import load_tcr

tv = load_tcr("data/example_tcr.pdb", traj="data/example_ensemble.xtc",
              legacy_anarci=False).pairs[0].traj
print("frames:", tv.n_frames)

## Features — Cα-distances and backbone dihedrals (per region)

In [ ]:
from kinapse.dynamics_analysis.embeddings.features import features_ca_dist, features_dihedrals

X_cadist = features_ca_dist(tv, ["A_CDR3"])          # (n_frames, n_Cα-pairs)
X_dihed  = features_dihedrals(tv, ["A_CDR3"])        # (n_frames, 2·n_dihedrals), sin/cos
print("CA-dist:", X_cadist.shape, " dihedrals:", X_dihed.shape)

## Reduce to 2-D and plot

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

Z = PCA(n_components=2).fit_transform(X_cadist)
plt.figure(figsize=(5, 4))
plt.scatter(Z[:, 0], Z[:, 1], s=10, c=range(len(Z)), cmap="viridis")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("A_CDR3 Cα-distance PCA"); plt.colorbar(label="frame")
plt.tight_layout()

## Free-energy surface (PMF) — `-kT ln p` on the 2-D embedding

In [ ]:
import numpy as np
from kinapse.dynamics_analysis.pmf_kde import compute_pmf_hist

xe = np.linspace(Z[:, 0].min(), Z[:, 0].max(), 30)
ye = np.linspace(Z[:, 1].min(), Z[:, 1].max(), 30)
fe, xedges, yedges = compute_pmf_hist(Z, xedges=xe, yedges=ye)   # free energy (kJ/mol)

plt.figure(figsize=(5, 4))
plt.pcolormesh(xedges, yedges, fe.T, cmap="viridis_r")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.colorbar(label="free energy (kJ/mol)")
plt.title("PMF of the A_CDR3 ensemble"); plt.tight_layout()

## Comparing two ensembles (the real workflow)
With a **ground-truth** and a **predicted** ensemble (each a `TrajectoryView`), the
end-to-end runners do align → featurize → reduce → PMF → **Jensen–Shannon divergence**
plus embedding-quality metrics (trustworthiness, Mantel), writing `*_metrics.json`:

```python
from kinapse.dynamics_analysis import run_ca_dist
run_ca_dist(tv_gt, tv_pred, outdir="out/A_CDR3", regions=["A_CDR3"], reducer="pca")
```
`kinapse.dynamics_analysis.rmsd_tm` gives per-frame RMSD / TM-score, and
`pmf_kde.oriol_analysis` orchestrates the hist+KDE PMFs and the JSD.